In [ ]:
using Plots
using NaturalNeighbours
using LinearAlgebra
using StableRNGs
using DelaunayTriangulation
using DataInterpolations
using RegularizationTools

In [ ]:
plotlyjs()
showGraphs = false;

In [ ]:
function read_custom_file(filepath::String)
    array1 = Float64[]
    array2 = Float64[]

    start_reading = false

    num_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    open(filepath, "r") do file
        for line in eachline(file)

            clean_line = strip(line)

            if startswith(clean_line, "+") || startswith(clean_line, "-")
                matches = [m.match for m in eachmatch(num_regex, clean_line)]

                if length(matches) >= 2
                    push!(array1, parse(Float64, matches[1]))
                    push!(array2, parse(Float64, matches[2]))
                end
            else
                continue
            end
        end
    end
    return array1, array2
end

In [ ]:
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/2.S9272.PtCoCu.7/S9272-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/6.S9281.CoPt.10/S9281-FORC-50-2000-5s"
cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/4.S9283.CoPt.6/S9283-FORC-50-1500-5s"
H_read = Float64[]
M_read = Float64[]
H_read, M_read = read_custom_file(cale * ".txt")

In [ ]:
## Normalize
normH = 1.0e3
normM = 1.0e-3
H_read = H_read ./ normH
M_read = M_read ./ normM
H_read, M_read

In [ ]:
if (showGraphs)
    scatter(H_read, M_read,
        title="Scatter Plot of Data",
        xlabel="H",
        ylabel="M",
        legend=false,
        markersize=3,
        markercolor=:indianred,
        markerstrokecolor=:indianred
    )
end

In [ ]:
## Detect FORCs (Hr, numPointsPerFORC)
Hr_read = Float64[]
numPointsPerFORC = Int[]

current_Hr = H_read[1]
push!(Hr_read, current_Hr)

counterPointsPerFORC = 1

for i in 2:(length(H_read))
    if H_read[i] < H_read[i-1]
        global current_Hr = H_read[i]
        push!(numPointsPerFORC, counterPointsPerFORC)
        global counterPointsPerFORC = 0
    end
    global counterPointsPerFORC += 1
    push!(Hr_read, current_Hr)
end
push!(numPointsPerFORC, counterPointsPerFORC)
numFORCs = length(numPointsPerFORC)

println("----------- Total $(length(numPointsPerFORC)) FORCs -----------")
println("---------  $(numPointsPerFORC[1]) first / $(numPointsPerFORC[numFORCs]) last ---------")

In [ ]:
## Save Hr-H-M original file
n = length(Hr_read)
if length(H_read) != n || length(M_read) != n
    error("Error: All three arrays must have the same length.")
end

open(cale * "_Hr-H-M_orig.dat", "w") do file
    for i in 1:n
        println(file, "$(Hr_read[i])  $(H_read[i])  $(M_read[i])")
    end
end

In [ ]:
function gimmeOneFORC(theFORC::Int64)
    contorNumPoints = 0
    H_interp = Float64[]
    M_interp = Float64[]
    #detect indexes for $(theFORC)
    if theFORC > 1
        for i in 1:theFORC-1
            contorNumPoints += numPointsPerFORC[i]
        end
    else
        contorNumPoints = 0
    end

    #myHr = Hr_read[contorNumPoints]
    startPointOnFORC = contorNumPoints + 1
    push!(H_interp, H_read[startPointOnFORC])
    push!(M_interp, M_read[startPointOnFORC])
    contorNumPoints = 1
    while (Hr_read[startPointOnFORC+contorNumPoints-1] - Hr_read[startPointOnFORC+contorNumPoints]) < 1.0e-5 #eps - compare floats
        push!(H_interp, H_read[startPointOnFORC+contorNumPoints])
        push!(M_interp, M_read[startPointOnFORC+contorNumPoints])
        contorNumPoints += 1
        if (startPointOnFORC + contorNumPoints) > length(Hr_read)
            break
        end
    end
    H_interp, M_interp
end

In [ ]:
points = [H_read'; Hr_read']
tri = triangulate(points)
#∇g = generate_gradients(tri, M_read)
;

In [ ]:
#_, Hg = generate_derivatives(tri, M_read)
_, Hg = generate_derivatives(tri, M_read, use_cubic_terms=false)
#_, Hg = generate_derivatives(tri, M_read, method=Iterative())

In [ ]:
myH = getindex.(Hg, 3)
#heatmap(myH)
;

In [ ]:
if (showGraphs)
    scatter(H_read, Hr_read, marker_z=myH, color=cgrad(:RdBu, scale=log10))
end

In [ ]:
open(cale * "_VoronoiFORC.dat", "w") do file
    for i in 1:n
        println(file, "$(H_read[i])  $(Hr_read[i])  $(-myH[i])")
    end
end

In [ ]:
numFORCs = length(numPointsPerFORC) + 1 # +1 pentru one-point-FORC

M_NoExtend = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendLine = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendMirror = zeros(Float64, numFORCs, numFORCs) #matricea M
Hr_NoExtend = zeros(Float64, numFORCs)
H_NoExtend = zeros(Float64, numFORCs)

H, M = gimmeOneFORC(1)
Hr_NoExtend[1] = H[end]
H_NoExtend[numFORCs] = H[end]
M_NoExtend[1, numFORCs] = M[end]
M_ExtendLine[1, numFORCs] = M[end]
M_ExtendMirror[1, numFORCs] = M[end]
for i in 1:numFORCs-1
    M_ExtendLine[1, i] = M[end]
    M_ExtendMirror[1, i] = M[end]
end

for i in 2:numFORCs
    H, M = gimmeOneFORC(i - 1)
    Hr_NoExtend[i] = H[begin]
    H_NoExtend[numFORCs-i+1] = H[begin]
end

In [ ]:
for i in 2:numFORCs
    t, u = gimmeOneFORC(i - 1)
    if t[end] < Hr_NoExtend[1]
        t[end] = Hr_NoExtend[1]
    end

    #Example = AkimaInterpolation(M_interp, H_interp)

    d = 3 + div(i, 3) - 1
    A = RegularizationSmooth(u, t, d; alg=:gcv_svd)
    û = A.û
    N = i #length(t)
    titp = [Hr_NoExtend[j] for j = i:-1:1] #collect(range(minimum(t), maximum(t), length=N))
    uitp = A.(titp)
    #println("---------- $(i)/$(numFORCs) -------------")

    for j in 1:i
        M_NoExtend[i, numFORCs-i+j] = uitp[j]
        M_ExtendLine[i, numFORCs-i+j] = uitp[j]
        M_ExtendMirror[i, numFORCs-i+j] = uitp[j]
        #println(" $(M_NoExtend[i, numFORCs-i+j]) ")
    end
    for j in 1:numFORCs-i
        M_ExtendLine[i, j] = uitp[begin]
    end
    for j in numFORCs-i+1:-1:1
        if j + (2 * (numFORCs - i + 1 - j) + 1) <= numFORCs
            M_ExtendMirror[i, j] = M_ExtendMirror[i, j+(2*(numFORCs-i+1-j)+1)]
        else
            M_ExtendMirror[i, j] = uitp[end]
        end
    end

end

In [ ]:
#heatmap(M_ExtendLine, yflip=true)

M_ExtendLine_Plot = zeros(Float64, numFORCs * numFORCs)
M_ExtendMirror_Plot = zeros(Float64, numFORCs * numFORCs)
Hr_NoExtend_Grid = zeros(Float64, div(numFORCs * (numFORCs + 1), 2))
H_NoExtend_Grid = zeros(Float64, div(numFORCs * (numFORCs+1), 2))
Hr_Extend_Grid = zeros(Float64, numFORCs * numFORCs)
H_Extend_Grid = zeros(Float64, numFORCs * numFORCs)
;

In [ ]:
contorPct = 1
for i = 1:numFORCs 
    for j = 1:i 
        Hr_NoExtend_Grid[contorPct] = Hr_NoExtend[i]
        H_NoExtend_Grid[contorPct] = H_NoExtend[numFORCs-i+j]
        contorPct += 1
    end
end

In [ ]:
itp = interpolate(H_read, Hr_read, M_read; derivatives=true)

In [ ]:
#sibson_vals = itp(_x, _y; method=Sibson())
#triangle_vals = itp(_x, _y; method=Triangle())
#laplace_vals = itp(_x, _y; method=Laplace())
sibson_1_vals = itp(H_NoExtend_Grid, Hr_NoExtend_Grid; method=Sibson(1));
#nearest_vals = itp(_x, _y; method=Nearest())
#farin_vals = itp(_x, _y; method=Farin())
hiyoshi_vals = itp(H_NoExtend_Grid, Hr_NoExtend_Grid; method=Hiyoshi(2));
;

In [ ]:
plot(H_NoExtend_Grid, Hr_NoExtend_Grid, sibson_1_vals; seriestype=:scatter)
#scatter(Hr_NoExtend_Grid, sibson_1_vals)


In [ ]:
open(cale * "_Sibson1FORC.dat", "w") do file
    for i in 1:div(numFORCs * (numFORCs + 1), 2)
        println(file, "$(H_NoExtend_Grid[i])  $(Hr_NoExtend_Grid[i])  $(sibson_1_vals[i]) $(hiyoshi_vals[i])")
    end
end